In [1]:
# ============================================================
# 6.1 ABSTRACT CORPUS + DICTIONARY MATCHING (FINAL)
# ============================================================

import json
import pandas as pd
import re
from collections import defaultdict
from tqdm import tqdm

from utils import (
    canonical_normalize_char,
    build_normalized_text_and_map,
    build_dictionary_lookup,
    find_exact_matches_raw,
    safe_to_list,
)


# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

LEXICON_FILE = "Dataset/VIOLIN_12-10-2025/interim/adjuvant_ner_lexicon_clean.csv"
ABSTRACT_FILE = "Dataset/VIOLIN_12-10-2025/interim/pubmed_abstracts.jsonl"

OUT_DIR = "Dataset/VIOLIN_12-10-2025/interim/exact_matches"

import os
os.makedirs("Dataset/VIOLIN_12-10-2025/interim/exact_matches", exist_ok=True)

OUT_JSONL = f"{OUT_DIR}/matched_exact_abstracts.jsonl"
OUT_CSV = f"{OUT_DIR}/matched_exact_adjuvants.csv"


# ------------------------------------------------------------
# STEP 1 — CANONICAL NORMALIZATION (CRITICAL)
# ------------------------------------------------------------


# ------------------------------------------------------------
# STEP 2 — LOAD LEXICON
# ------------------------------------------------------------

lex = pd.read_csv(LEXICON_FILE, encoding="utf-8-sig")

import ast

from collections import defaultdict
from utils import build_dictionary_lookup, safe_to_list

lex = pd.read_csv(LEXICON_FILE, encoding="utf-8-sig")
lex["synonyms"] = lex["synonyms"].apply(safe_to_list)
lex["expanded_forms"] = lex["expanded_forms"].apply(safe_to_list)

surface_forms = defaultdict(set)  # raw lowercase terms -> VO IDs
for _, row in lex.iterrows():
    vo = row["adjuvant_vo_id"]
    if isinstance(row.get("preferred_name"), str) and row["preferred_name"].strip():
        surface_forms[row["preferred_name"].strip().lower()].add(vo)
    for col in ("synonyms", "expanded_forms"):
        for s in row[col]:
            if isinstance(s, str) and s.strip():
                surface_forms[s.strip().lower()].add(vo)

print(f"Loaded {len(surface_forms)} dictionary surface forms")

# ------------------------------------------------------------
# STEP 3 — MATCHING FUNCTION (CORRECT OFFSETS)
# ------------------------------------------------------------

'''def find_dictionary_matches(raw_text: str):
    return find_exact_matches_raw(raw_text, surface_forms)'''
def find_dictionary_matches(raw_text: str):
    return find_exact_matches_raw(raw_text, surface_forms)


# ------------------------------------------------------------
# STEP 4 — RUN OVER ABSTRACT CORPUS
# ------------------------------------------------------------

abstract_results = []
mention_rows = []

with open(ABSTRACT_FILE, encoding="utf-8") as f:
    for line in tqdm(f, desc="Exact+Expanded matching"):
        rec = json.loads(line)

        pmid = str(rec.get("pmid", "")).strip()
        abstract = rec.get("abstract", "")

        if not abstract or not pmid:
            continue

        matches = find_dictionary_matches(abstract)

        if not matches:
            continue

        unique_vos = sorted({m["adjuvant_vo_id"] for m in matches})

        abstract_results.append({
            "pmid": pmid,
            "abstract": abstract,
            "matches": matches,
            "num_mentions": len(matches),
            "num_unique_adjuvants": len(unique_vos)
        })

        for m in matches:
            mention_rows.append({
                "pmid": pmid,
                "adjuvant_vo_id": m["adjuvant_vo_id"],
                "matched_text": m["matched_text"],
                "start": m["start"],
                "end": m["end"],
                "match_type": m["match_type"]
            })

# ------------------------------------------------------------
# STEP 5 — SAVE OUTPUTS
# ------------------------------------------------------------

pd.DataFrame(mention_rows).to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

with open(OUT_JSONL, "w", encoding="utf-8") as out:
    for r in abstract_results:
        out.write(json.dumps(r, ensure_ascii=False) + "\n")

# ------------------------------------------------------------
# STEP 6 — STATS
# ------------------------------------------------------------

print("\n✅ EXACT + EXPANDED MATCHING COMPLETE")
print(f"• Total mentions found: {len(mention_rows)}")
print(f"• Unique PMIDs matched: {len(set(r['pmid'] for r in abstract_results))}")
print(f"• Unique adjuvant VO IDs matched: {len(set(r['adjuvant_vo_id'] for r in mention_rows))}")
print(f"• Output CSV: {OUT_CSV}")
print(f"• Output JSONL: {OUT_JSONL}")


Loaded 363 dictionary surface forms


Exact+Expanded matching: 0it [00:00, ?it/s]

Exact+Expanded matching: 7it [00:00, 64.23it/s]

Exact+Expanded matching: 15it [00:00, 67.85it/s]

Exact+Expanded matching: 25it [00:00, 80.21it/s]

...

Exact+Expanded matching: 1325it [00:18, 72.06it/s]

Exact+Expanded matching: 1333it [00:18, 68.62it/s]

Exact+Expanded matching: 1341it [00:18, 71.09it/s]

Exact+Expanded matching: 1341it [00:18, 72.41it/s]


✅ EXACT + EXPANDED MATCHING COMPLETE
• Total mentions found: 1031
• Unique PMIDs matched: 281
• Unique adjuvant VO IDs matched: 56
• Output CSV: Dataset/VIOLIN_12-10-2025/interim/exact_matches/matched_exact_adjuvants.csv
• Output JSONL: Dataset/VIOLIN_12-10-2025/interim/exact_matches/matched_exact_abstracts.jsonl
